<img src="https://raw.githubusercontent.com/Nextia-Intelligence/nextia-learning-labs/main/nextia-notebook-branding/assets/courses/c39-header.png" alt="Nextia Learning · Entity Resolution and Record Matching: Is This the Same Thing?" width="100%">

# The golden record

**[The golden record](https://learning.nextia-ai.com/courses/record-matching/m07/the-golden-record/)** · Entity Resolution and Record Matching: Is This the Same Thing? · Module 7, lesson 3 of 4 · [learning.nextia-ai.com](https://learning.nextia-ai.com)

**You will learn to** build one golden record for each entity with a merge rule per field, keep the source of every value, and undo a merge.

| | |
|---|---|
| **Time** | About 40 minutes |
| **Needs** | An internet connection for the setup cells (about 6 MB). |
| **You should know** | Safe clustering, from [Safer clustering](https://learning.nextia-ai.com/courses/record-matching/m07/safer-clustering/). |
| **Tested** | Python 3.12 with NumPy 2.5.4, pandas 3.1.1 and matplotlib 3.11.2 (rapidfuzz 3.14.6, jellyfish 1.3.0, datasketch 2.0.0, scikit-learn 1.9.1 and LightGBM 4.7.0 where the lesson needs them), in Jupyter on a laptop CPU (Apple M4, 16 GB), from a fresh start (*Restart and run all*) |

### How to use this notebook

This notebook has every query, task and check of the lesson. The [lesson page](https://learning.nextia-ai.com/courses/record-matching/m07/the-golden-record/) has the full explanations, the diagrams and the knowledge checks that count toward your certificate. Keep both open.

1. Run the cells in order, from the top. Press **Shift+Enter** to run a cell and move to the next one.
2. At a **Predict** note, write your prediction down before you run the cell.
3. At a **Your turn** note, write your answer in the cell, run it, then run the check cell below it.
4. If you are stuck, open the [solution notebook](https://github.com/Nextia-Intelligence/nextia-learning-labs/blob/main/C39/M07-L03-the-golden-record/the-golden-record-solution.ipynb).

## Setup: install the packages

Run the next cell. It installs the versions that the course uses: numpy 2.5.4, pandas 3.1.1, and matplotlib for the pictures. It needs an internet connection and takes about a minute the first time.

You should see one line per package, with the versions above. If you see other versions, restart the kernel (in Colab: **Runtime › Restart session**; in Kaggle: **Run › Restart & clear cell outputs**), then run this cell again.

In [ ]:
%pip install -q --disable-pip-version-check numpy==2.5.4 pandas==3.1.1 matplotlib==3.11.2
import warnings; warnings.filterwarnings("ignore")
from importlib.metadata import version
for name in ['numpy', 'pandas']:
    print(name, version(name))

## Setup: get the Larkfield data

Run the next cell. It downloads the files that this lesson needs (about 8 MB) into a folder `larkfield-match` and checks each file's checksum. It needs an internet connection and no account, and takes about 20 seconds.

You should see `Ready:` and the number of files. If you run the cell again, it uses the files that are already there.

If the cell shows an error, read its last line:

- `Could not download`: check your internet connection, then run the cell again. In Kaggle, turn on **Internet** in the notebook settings.
- `wrong checksum`: delete the folder `larkfield-match` (in Colab: the **Files** panel on the left), then run the cell again.

Where the data comes from: Larkfield is an invented shop. Every customer record and every product listing is **synthetic**: our code made them from invented people and products, and then added realistic errors on purpose (typing errors, nicknames, old addresses, shared family emails). Because the code knows the true person behind each record, you can always check an answer. The [dataset card](https://github.com/Nextia-Intelligence/nextia-learning-labs/blob/main/C39/data/dataset-card.md) explains the error model.

In [ ]:
# Setup: download the files into larkfield-match/ and check them.
import hashlib, os, shutil, subprocess, urllib.request
from pathlib import Path

LABS = "https://raw.githubusercontent.com/Nextia-Intelligence/nextia-learning-labs/main/C39/"
FILES = {
    "data/sources/shop_accounts.csv": "f59f3a9619c35b572650e10663a0ba5f316202d173850689ec9242706fb4174e",
    "data/sources/loyalty_members.csv": "629696f38469898be48cb664e5d69f9c6d438e8622899bbb281bd520bf44af32",
    "data/sources/garden_centres.csv": "2878063d24a38d841f93bb14e07b2d46619fcc4a3909e92ce0238116434d85b2",
    "data/truth.csv": "d951378129bdf30b649250e6d2127787e5630d66aa7cfe5793b1a110e37a06d3",
}
HERE = Path.cwd()
# The notebook works in its own folder, marked with a file, so it never
# writes into your own project if you open the notebook there.
if Path(".nextia-notebook").exists():  # the cell ran before: start from the parent folder
    os.chdir("..")
    HERE = Path.cwd()
PROJECT = Path("larkfield-match").resolve()
(PROJECT / "data").mkdir(parents=True, exist_ok=True)
(PROJECT / ".nextia-notebook").touch()

def sha256(path):
    return hashlib.sha256(path.read_bytes()).hexdigest()

def download(url, path):
    try:
        urllib.request.urlretrieve(url, path)
    except OSError:
        # Python without certificates (python.org on macOS): try curl.
        if subprocess.run(["curl", "-fsSL", "-o", str(path), url]).returncode:
            raise SystemExit(f"Could not download {url}. Check your internet connection.")

for name, expected in FILES.items():
    path = PROJECT / name
    path.parent.mkdir(parents=True, exist_ok=True)
    if not (path.exists() and sha256(path) == expected):
        local = HERE / name  # a copy next to the notebook, if there is one
        if local.exists():
            shutil.copy(local, path)
        else:
            download(LABS + name, path)
        if sha256(path) != expected:
            raise SystemExit(f"{name} has the wrong checksum. Delete the folder larkfield-match and run again.")

os.chdir(PROJECT)  # work in the project folder, as in the lesson
print(f"Ready: {len(FILES)} files in larkfield-match.")

## Setup: helpers

The next cell loads NumPy as `np`, pandas as `pd` and matplotlib. It defines a few small helpers:

- `source(name)` reads one customer source as it is (`"shop"`, `"loyalty"` or `"garden"`); `records()` reads all three in shared columns (`record_id, source, name, email, phone, address, postcode, date_of_birth`), with the values exactly as the sources wrote them.
- `truth()` says which true customer (`entity_id`) each record belongs to, and its split (`train`, `validation` or `test`).
- `pair_measures(pairs)` measures a table of pairs (`record_id_l`, `record_id_r`) that a matcher calls "the same": pair precision, recall and F1 on the test split, as in the project's `match eval`.

It also defines the check helpers. They tell you if your answer is right, without showing the answer:

- `expect(what, yours, expected)` compares one of your numbers with the expected one, and says "too high" or "too low".
- `expect_hash(what, yours, expected)` compares any result with a hidden fingerprint of the expected one.

Run the cell. It prints nothing.

In [ ]:
# Helpers: the data, pair measures, and checks that do not show the answer.
import hashlib, json, warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
warnings.filterwarnings("ignore")

FILES_OF = {"shop": "shop_accounts.csv", "loyalty": "loyalty_members.csv", "garden": "garden_centres.csv"}

def source(name):
    """One customer source as it is: its own columns, every value as text."""
    return pd.read_csv(f"data/sources/{FILES_OF[name]}", dtype=str, keep_default_na=False)

def records():
    """All customer records in shared columns, values as the sources wrote them."""
    s, l, g = source("shop"), source("loyalty"), source("garden")
    parts = [
        pd.DataFrame({"record_id": s.record_id, "source": "shop", "name": s.first_name + " " + s.last_name,
                      "email": s.email, "phone": s.phone, "address": s.address + ", " + s.city + " " + s.province,
                      "postcode": s.postcode, "date_of_birth": s.date_of_birth}),
        pd.DataFrame({"record_id": l.record_id, "source": "loyalty", "name": l.full_name, "email": l.email,
                      "phone": l.phone, "address": "", "postcode": l.postcode, "date_of_birth": l.date_of_birth}),
        pd.DataFrame({"record_id": g.record_id, "source": "garden", "name": g.name, "email": g.email,
                      "phone": g.phone, "address": g.address, "postcode": "", "date_of_birth": g.date_of_birth}),
    ]
    return pd.concat(parts, ignore_index=True)

def truth():
    """record_id -> entity_id (the true customer) and split."""
    return pd.read_csv("data/truth.csv", dtype=str)

def ordered(pairs):
    """The same pairs with the smaller record ID first, one row per pair."""
    lo = pairs[["record_id_l", "record_id_r"]].min(axis=1)
    hi = pairs[["record_id_l", "record_id_r"]].max(axis=1)
    out = pd.DataFrame({"record_id_l": lo, "record_id_r": hi})
    return out[out.record_id_l != out.record_id_r].drop_duplicates()

def true_pairs():
    """Every pair of records of the same true customer, with its split."""
    t = truth()
    m = t.merge(t, on="entity_id", suffixes=("_l", "_r"))
    m = m[m.record_id_l < m.record_id_r]
    return m[["record_id_l", "record_id_r", "split_l"]].rename(columns={"split_l": "split"})

RANK = {"train": 0, "validation": 1, "test": 2}

def pair_measures(pairs, split="test"):
    """Pair precision, recall and F1 of the pairs a matcher calls the same (a pair is in the test split when one
    of its records belongs to a test customer)."""
    p = ordered(pairs)
    t = truth().set_index("record_id")
    rank = t.split.map(RANK)
    p = p.assign(split=np.maximum(p.record_id_l.map(rank), p.record_id_r.map(rank)).map({v: k for k, v in RANK.items()}),
                 same=(p.record_id_l.map(t.entity_id) == p.record_id_r.map(t.entity_id)))
    tp_all = true_pairs()
    p, tp_all = p[p.split == split], tp_all[tp_all.split == split]
    tp = int(p.same.sum())
    precision = tp / len(p) if len(p) else 0.0
    recall = tp / len(tp_all) if len(tp_all) else 0.0
    f1 = 2 * precision * recall / (precision + recall) if precision + recall else 0.0
    return {"precision": round(precision, 3), "recall": round(recall, 3), "f1": round(f1, 3),
            "pairs": len(p), "right": tp, "true_pairs": len(tp_all)}

def same_value_pairs(df, col):
    """Every pair of records that share a non-empty value of col."""
    d = df[df[col].notna() & (df[col] != "")][["record_id", col]]
    sizes = d[col].value_counts()
    n_pairs = int((sizes * (sizes - 1) // 2).sum())
    if n_pairs > 20_000_000:   # protect the computer's memory from a key that is too wide
        raise ValueError(f"This key makes {n_pairs:,} pairs: too many to hold in memory. Use a narrower key.")
    m = d.merge(d, on=col, suffixes=("_l", "_r"))
    return m[m.record_id_l < m.record_id_r][["record_id_l", "record_id_r"]].drop_duplicates()

def fingerprint(value):
    def norm(v):
        if hasattr(v, "tolist"):
            return norm(v.tolist())
        if isinstance(v, dict):
            return {str(k): norm(x) for k, x in sorted(v.items(), key=lambda kv: str(kv[0]))}
        if isinstance(v, (list, tuple)):
            return [norm(x) for x in v]
        if isinstance(v, bool) or v is None or isinstance(v, int):
            return v
        if isinstance(v, float):
            return None if v != v else round(v, 3)
        if hasattr(v, "item"):
            return norm(v.item())
        return str(v)
    return hashlib.sha256(json.dumps(norm(value), sort_keys=True).encode()).hexdigest()[:16]

def expect(what, yours, expected, tolerance=1e-6):
    """Compare one number or value with the expected one."""
    if isinstance(yours, (np.integer, np.floating)):
        yours = yours.item()
    if yours is None or yours is Ellipsis:
        print(f"Not yet: {what} has no value. Write your answer in the cell above, then run it again.")
        return
    if isinstance(expected, float) and isinstance(yours, (int, float)):
        same = abs(yours - expected) <= tolerance * max(1, abs(expected))
    else:
        same = yours == expected
    if same:
        print(f"Check passed: {what} is {yours}.")
    elif isinstance(expected, (int, float)) and isinstance(yours, (int, float)):
        print(f"Not yet: {what} is {yours}, which is too {'high' if yours > expected else 'low'}.")
    else:
        print(f"Not yet: {what} is {yours!r}, which is not the expected value.")

def expect_hash(what, yours, expected):
    """Compare any result with the fingerprint of the expected one."""
    if yours is None or yours is Ellipsis:
        print(f"Not yet: {what} has no value. Write your answer in the cell above, then run it again.")
    elif fingerprint(yours) == expected:
        print(f"Check passed: {what} is right.")
    else:
        print(f"Not yet: {what} is not the expected result. Read the task again, and check each step.")

<img src="https://raw.githubusercontent.com/Nextia-Intelligence/nextia-learning-labs/main/nextia-notebook-branding/assets/section-divider-light.png" alt="" width="100%">

## Setup: the clean records of Module 2

The next cell cleans every record as Module 2 did: names without accents and titles, split into first name and
surname; emails in lower case; phones as 10 digits; postcodes as 6 characters; dates as YYYY-MM-DD. Run it. It
prints the number of records.

In [ ]:
# The clean records (Module 2): one row per record with first, last, email, phone, postcode, dob.
import re, unicodedata

def strip_accents(s):
    return "".join(c for c in unicodedata.normalize("NFKD", s) if not unicodedata.combining(c))

def norm_text(s):
    s = strip_accents(str(s or "")).lower().replace("'", "")
    return " ".join(w for w in re.sub(r"[^\w\s]", " ", s).split() if w not in {"mr", "mrs", "ms", "dr", "m", "mme"})

def norm_phone(s):
    d = re.sub(r"\D", "", re.split(r"(?i)\s*ext", str(s or ""))[0])
    d = d[1:] if len(d) == 11 and d.startswith("1") else d
    return d if len(d) == 10 else ""

def norm_postcode(s):
    p = re.sub(r"[\s-]", "", str(s or "")).upper()
    return p if re.fullmatch(r"[A-Z]\d[A-Z]\d[A-Z]\d", p) else ""

def norm_date(s):
    s = str(s or "")
    m = re.fullmatch(r"(\d{1,2})/(\d{1,2})/(\d{4})", s)
    return f"{m[3]}-{int(m[2]):02d}-{int(m[1]):02d}" if m else (s if re.fullmatch(r"\d{4}-\d\d-\d\d", s) else "")

def split_name(s):
    s = str(s or "")
    if "," in s:
        last, first = s.split(",", 1)
    else:
        words = s.split()
        first, last = " ".join(words[:-1]), (words[-1] if words else "")
    return norm_text(first), norm_text(last)

def postcode_of(postcode, address):
    """The postcode column, or the postcode at the end of a garden-centre address."""
    found = re.findall(r"[A-Za-z]\d[A-Za-z][ -]?\d[A-Za-z]\d", address)
    return norm_postcode(postcode) or (norm_postcode(found[-1]) if found else "")

recs = records()
names = recs.name.map(split_name)
clean = pd.DataFrame({
    "record_id": recs.record_id, "source": recs.source,
    "first": [f for f, _ in names], "last": [l.replace(" ", "") for _, l in names],
    "email": recs.email.str.strip().str.lower(), "phone": recs.phone.map(norm_phone),
    "postcode": [postcode_of(p, a) for p, a in zip(recs.postcode, recs.address)],
    "dob": recs.date_of_birth.map(norm_date),
})
print(f"{len(clean):,} clean records")

## Setup: the matched pairs

The next cell makes the pairs that Module 2's four rules call the same: same email; same phone and surname; same name and date of birth; same name and postcode. Each pair gets a **score**: the number of fields that agree, from 1 to 6. A higher score is a stronger link. Run it. It prints the pairs and their pair measures.

In [ ]:
# The pairs of the four rules (Module 2), with a score: how many of six fields agree.
def key_pairs(cols):
    keep = (clean[cols] != "").all(axis=1)
    d = clean[keep].assign(key=clean[keep][cols].agg("|".join, axis=1))
    return same_value_pairs(d, "key")

rules = pd.concat([key_pairs(["email"]), key_pairs(["phone", "last"]), key_pairs(["first", "last", "dob"]),
                   key_pairs(["first", "last", "postcode"])]).drop_duplicates().reset_index(drop=True)
c = clean.set_index("record_id")
FIELDS = ["first", "last", "email", "phone", "postcode", "dob"]
score = 0
for f in FIELDS:
    a, b = rules.record_id_l.map(c[f]), rules.record_id_r.map(c[f])
    score = score + ((a == b) & (a != "")).astype(int)
rules["score"] = score
print(f"{len(rules):,} pairs called the same;", pair_measures(rules))

## Setup: the clusters of lessons 1 and 2

The next cell joins the pairs into clusters with connected components (lesson 1), then makes them safe (lesson 2): while a cluster holds two different dates of birth or more than 6 records, it cuts the weakest link. Run it. It prints both results.

In [ ]:
# Connected components (lesson 1): every record joined by a chain of pairs ends in one cluster.
def components(pairs, ids):
    parent = {r: r for r in ids}
    def find(x):
        while parent[x] != x:
            parent[x] = parent[parent[x]]
            x = parent[x]
        return x
    for a, b in zip(pairs.record_id_l, pairs.record_id_r):
        ra, rb = find(a), find(b)
        if ra != rb:
            parent[max(ra, rb)] = min(ra, rb)
    return pd.Series({r: find(r) for r in ids}, name="cluster")

ids = clean.record_id.tolist()
cc = components(rules, ids)
sizes = cc.value_counts()
print(f"connected components: {cc.nunique():,} entities (true: 37,500); largest clusters:", sizes.head(5).tolist())

def dob_conflict(dobs):
    """True when the non-empty dates hold two different dates of birth. A day and month swap is the same date."""
    seen = set()
    for d in dobs:
        if not d:
            continue
        swapped = d[:4] + "-" + d[8:10] + "-" + d[5:7]
        if seen and d not in seen and swapped not in seen:
            return True
        seen.add(d)
    return False

# Safe clustering (lesson 2): while a cluster fails a check, cut its weakest link.
MAX_SIZE = 6
dob_of = dict(zip(clean.record_id, clean.dob))

def problems(members):
    why = []
    if len(members) > MAX_SIZE:
        why.append("too many records")
    if dob_conflict([dob_of[r] for r in members]):
        why.append("two dates of birth")
    return why

def pieces(nodes, links):
    """The connected parts of one cluster, after some links were cut."""
    nb = {n: set() for n in nodes}
    for a, b in links:
        nb[a].add(b); nb[b].add(a)
    left, out = set(nodes), []
    while left:
        stack = [left.pop()]; part = set(stack)
        while stack:
            for m in nb[stack.pop()]:
                if m in left:
                    left.remove(m); part.add(m); stack.append(m)
        out.append(part)
    return out

def safe_clusters(pairs, clusters, cuts):
    assign = clusters.to_dict()
    groups = clusters.groupby(clusters).groups
    links = pairs.assign(c=pairs.record_id_l.map(clusters))
    bad = [cid for cid, members in groups.items() if len(members) > 1 and problems(list(members))]
    by_cluster = dict(list(links[links.c.isin(bad)].groupby("c")))
    for cid in bad:
        e = by_cluster[cid]
        weight = {(a, b): s for a, b, s in zip(e.record_id_l, e.record_id_r, e.score)}
        stack = [set(groups[cid])]
        while stack:
            part = stack.pop()
            why = problems(part)
            if len(part) == 1 or not why:
                for r in part:
                    assign[r] = min(part)
                continue
            inside = {k: w for k, w in weight.items() if k[0] in part and k[1] in part}
            weakest = min(inside, key=lambda k: (inside[k], k))
            del weight[weakest]
            cuts.append((*weakest, inside[weakest], ", ".join(why)))
            stack.extend(pieces(part, [k for k in weight if k[0] in part and k[1] in part]))
    return pd.Series(assign, name="cluster")

cuts = []
safe = safe_clusters(rules, cc, cuts)
print(f"safe clustering: {safe.nunique():,} entities, largest clusters:", safe.value_counts().head(5).tolist(),
      f"; {len(cuts):,} links cut")

## 1. One entity, several records

Each entity now has one or more records, and they do not agree on every field. Run the next cell. It puts every record next to its entity and its **creation date** (when the customer opened the account, joined the loyalty programme or first visited a garden centre). It sorts each entity's records with the newest first, and it shows the 3 records of Valérie Tremblay's entity.

In [ ]:
s, l, g = source("shop"), source("loyalty"), source("garden")
created = pd.concat([pd.Series(s.created.values, index=s.record_id), pd.Series(l.joined.values, index=l.record_id),
                     pd.Series(g.first_visit.values, index=g.record_id)])
TRUST = {"shop": 0, "loyalty": 1, "garden": 2}
table = clean.assign(full=(clean["first"] + " " + clean["last"]).str.strip(), created=clean.record_id.map(created),
                     trust=clean.source.map(TRUST), entity=clean.record_id.map(safe))
table = table.sort_values(["entity", "created"], ascending=[True, False])   # newest record first in each entity
valerie = table[table.entity == safe["G-000001"]]
print(valerie[["record_id", "source", "full", "email", "phone", "postcode", "dob", "created"]].to_string(index=False))

## 2. Merge rules, one per field

A **golden record** takes one value for each field. A **merge rule** says which value wins. Our rules (the project uses the same ones):

| Field | Rule | Why |
|---|---|---|
| name | the most common full name; on a tie, the most trusted source | a name is typed in many ways; the shop's sign-up form is the most careful |
| email, phone, postcode | the newest record that has one | people change them; the newest is most likely to work |
| date of birth | the most common one | it never changes, so a different value is a typing error |

Run the next cell: it defines the rules for the name and the date of birth.

In [ ]:
from collections import Counter

def most_common_name(g):
    """The most common full name; on a tie, the most trusted source (shop, loyalty, garden), then the newest."""
    names = Counter(g.full[g.full != ""])
    top = max(names.values())
    cand = g[g.full.isin([n for n, c in names.items() if c == top])].sort_values(["trust", "created"],
                                                                                ascending=[True, False])
    return cand.full.iloc[0], cand.record_id.iloc[0]

def most_common_dob(g):
    dobs = Counter(d for d in g.dob if d)
    if not dobs:
        return "", ""
    d = dobs.most_common(1)[0][0]
    return d, g[g.dob == d].record_id.iloc[0]

## 3. Your turn: the newest value

Complete `newest(g, field)`. The records of `g` are sorted with the newest first, so the answer is the first record that has a value. Return the value **and** the record it came from: a golden record must say where each value came from.

In [ ]:
def has_answer(f):
    try:
        f(valerie, "email")
        return True
    except Exception:
        return False

In [ ]:
def newest(g, field):
    """The value of field in the newest record that has one, and that record's ID ("" when no record has one).
    g is sorted with the newest record first."""
    has = ...       # the rows of g where field is not empty
    if len(has) == 0:
        return "", ""
    return ...      # (the value of field in the first row of has, the record_id of that row)

answer = [newest(valerie, f) for f in ("email", "phone", "postcode")] if has_answer(newest) else None
print(answer)

In [ ]:
expect_hash('answer', answer, '5d1ef436af299c81')

## 4. Build every golden record

Run the next cell. It builds Valérie's golden record, then the golden record of every entity that has more than one record.

In [ ]:
if not has_answer(newest):      # the task above is not done yet: the later cells still run, with empty values
    print("Complete newest() in the task above. Until then, email, phone and postcode stay empty.")
    def newest(g, field):
        return "", ""

def golden_row(g):
    row = {"entity": g.entity.iloc[0], "records": len(g), "sources": ",".join(sorted(set(g.source)))}
    row["name"], row["name_from"] = most_common_name(g)
    for f in ("email", "phone", "postcode"):
        row[f], row[f + "_from"] = newest(g, f)
    row["dob"], row["dob_from"] = most_common_dob(g)
    return row

print(pd.Series(golden_row(valerie)))
multi = table[table.entity.map(table.entity.value_counts()) > 1]
golden = pd.DataFrame([golden_row(g) for _, g in multi.groupby("entity", sort=True)])
print(f"{safe.nunique():,} entities; {len(golden):,} have more than one record and need a golden record")

In [ ]:
expect('golden records', len(golden), 16071)

> **Check.** 16,071 entities have more than one record. Every value has a `_from` column: the record it came from. The source records do not change.

Which source wins each field? Run the next cell.

In [ ]:
share = {f: (golden[f + "_from"].str[0].value_counts(normalize=True).round(3).to_dict()) for f in
         ("name", "email", "phone", "postcode", "dob")}
print("Which source gave each field (S shop, L loyalty, G garden centres):")
print(pd.DataFrame(share).fillna(0).T)

> **Predict.** The loyalty records are often the newest (customers joined the programme after they opened a shop account). Which fields will they win most often?

## 5. Undo a merge

Ivana finds a wrong merge. We take one record out. It gets its own entity. We write the change in a history, and we remember that the record may never join the others again (a **cannot-link** pair). Run the next cell: it takes Valérie's shop record out, only to show the mechanism.

In [ ]:
history, cannot_link = [], []

def undo(record):
    """Take one record out of its entity. It gets its own entity, and it may never join the others again."""
    old = safe[record]
    others = safe.index[(safe == old) & (safe.index != record)].tolist()
    safe[record] = record
    if old == record:                      # the entity was named after this record: the others get a new name
        safe[others] = min(others)
    cannot_link.extend((min(record, o), max(record, o)) for o in others)
    history.append({"event": "split", "record": record, "from_entity": old, "new_entity": safe[record]})

undo("S-009232")
print(history[-1])
print("cannot link:", cannot_link)
print(safe[["G-000001", "L-010072", "S-009232"]].to_dict())

The project does the same with `match undo`. It writes the history to `outputs/history.jsonl` and the cannot-link pairs to `outputs/cannot_link.csv`, and safe clustering reads them, so a new run does not merge the records again.

```sh
match golden
match golden --show LK-G-000001
match undo S-009232
```

<img src="https://raw.githubusercontent.com/Nextia-Intelligence/nextia-learning-labs/main/nextia-notebook-branding/assets/section-divider-light.png" alt="" width="100%">

**Next lesson:** [Cluster measures](https://learning.nextia-ai.com/courses/record-matching/m07/cluster-measures/) · **This lesson:** [read it on the site](https://learning.nextia-ai.com/courses/record-matching/m07/the-golden-record/)

<img src="https://raw.githubusercontent.com/Nextia-Intelligence/nextia-learning-labs/main/nextia-notebook-branding/assets/notebook-footer-light.png" alt="Nextia Learning · learning.nextia-ai.com" width="100%">

Free, practical AI courses at [learning.nextia-ai.com](https://learning.nextia-ai.com). Every customer record and every product listing here is synthetic: our code made them from invented people and products. No real person is in this data.